In [2]:
!pip install geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.6/323.6 kB 774.5 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 690.4 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 658.4 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 628.7 kB/s eta 0:00:0000:0100:01


In [4]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Load the outdoor temperature data
outdoor_temp = pd.read_csv('./campaign_data/outdoor_temp.csv', dtype={'gps_lat_out': str, 'gps_long_out': str})

# Convert 'Date/Time' to datetime format
outdoor_temp['Date/Time'] = pd.to_datetime(outdoor_temp['Date/Time'])

# Filter out rows where Lat or Lon is NaN
outdoor_temp = outdoor_temp.dropna(subset=['gps_lat_out', 'gps_long_out'])

# Convert lat/lon to float
outdoor_temp['gps_lat_out'] = outdoor_temp['gps_lat_out'].astype(float)
outdoor_temp['gps_long_out'] = outdoor_temp['gps_long_out'].astype(float)

# Load the shapefile
ama_boundary = gpd.read_file('shapefiles/AMA.shp')

# Create a GeoDataFrame for the outdoor temperature data
geometry = [Point(xy) for xy in zip(outdoor_temp['gps_long_out'], outdoor_temp['gps_lat_out'])]
outdoor_temp_gdf = gpd.GeoDataFrame(outdoor_temp, geometry=geometry)

# Ensure both GeoDataFrames have the same coordinate reference system (CRS)
outdoor_temp_gdf = outdoor_temp_gdf.set_crs(ama_boundary.crs, allow_override=True)

# Perform the spatial join to filter points within the polygon
outdoor_temp_within_boundary = gpd.sjoin(outdoor_temp_gdf, ama_boundary, predicate='within')

# Drop the geometry column to return to a standard DataFrame
outdoor_temp_within_boundary = pd.DataFrame(outdoor_temp_within_boundary.drop(columns='geometry'))

# Display the filtered data
print(outdoor_temp_within_boundary)


/tmp/ipykernel_13074/3061527345.py:6: DtypeWarning: Columns (7,10,11) have mixed types. Specify dtype option on import or set low_memory=False.
  outdoor_temp = pd.read_csv('./campaign_data/outdoor_temp.csv', dtype={'gps_lat_out': str, 'gps_long_out': str})


        point_id_outdoor_temp           Date/Time Unit  Value  \
0                AD_3F6B_8704 2022-08-05 09:00:00    C   28.0   
1                AD_3F6B_8704 2022-08-05 09:10:00    C   25.5   
2                AD_3F6B_8704 2022-08-05 09:20:00    C   25.5   
3                AD_3F6B_8704 2022-08-05 09:30:00    C   26.0   
4                AD_3F6B_8704 2022-08-05 09:40:00    C   26.0   
...                       ...                 ...  ...    ...   
1334677              UGH_6D23 2023-09-25 06:31:00    C   23.5   
1334678              UGH_6D23 2023-09-25 06:41:00    C   24.0   
1334679              UGH_6D23 2023-09-25 06:51:00    C   24.0   
1334680              UGH_6D23 2023-09-25 07:01:00    C   24.0   
1334681              UGH_6D23 2023-09-25 07:11:00    C   30.0   

        sensor_id_outdoor_temp site_id  \
0                       '3F6B'      AD   
1                       '3F6B'      AD   
2                       '3F6B'      AD   
3                       '3F6B'      AD   
4        

In [7]:
print(outdoor_temp['site_id'].unique())
print(outdoor_temp_within_boundary['site_id'].unique())

['AD' 'ASH' 'EL' 'JT' 'LA' 'N1W' 'NM' 'S0' 'S100' 'S101' 'S102' 'S103'
 'S104' 'S105' 'S106' 'S107' 'S108' 'S109' 'S10' 'S110' 'S111' 'S112'
 'S113' 'S114' 'S115' 'S116' 'S117' 'S118' 'S119' 'S11' 'S120' 'S121'
 'S122' 'S123' 'S124' 'S125' 'S126' 'S129' 'S12' 'S130' 'S131' 'S132'
 'S133' 'S134' 'S135' 'S136' 'S138' 'S139' 'S13' 'S143' 'S146' 'S147'
 'S14' 'S150' 'S153' 'S154' 'S159' 'S160' 'S165' 'S166' 'S168' 'S18' 'S19'
 'S1' 'S20' 'S21' 'S22' 'S24' 'S25' 'S26' 'S27' 'S28' 'S29' 'S30' 'S31'
 'S33' 'S34' 'S35' 'S36' 'S37' 'S38' 'S39' 'S3' 'S40' 'S41' 'S42' 'S43'
 'S44' 'S45' 'S47' 'S49' 'S4' 'S50' 'S51' 'S52' 'S53' 'S54' 'S55' 'S56'
 'S57' 'S58' 'S59' 'S5' 'S60' 'S61' 'S62' 'S64' 'S65' 'S66' 'S67' 'S68'
 'S69' 'S6' 'S70' 'S71' 'S72' 'S73' 'S74' 'S76' 'S77' 'S78' 'S79' 'S7'
 'S81' 'S82' 'S83' 'S85' 'S86' 'S87' 'S88' 'S89' 'S8' 'S90' 'S91' 'S92'
 'S93' 'S94' 'S96' 'S97' 'S98' 'TF' 'TMW' 'UGH']
['AD' 'EL' 'JT' 'N1W' 'NM' 'S100' 'S101' 'S102' 'S103' 'S104' 'S105'
 'S106' 'S107' 'S108' 'S1

In [6]:
outdoor_temp_within_boundary.to_csv('./campaign_data/outdoor_temp_AMA.csv')